In [ ]:
# SPDX-License-Identifier: MIT
# Copyright 2025-2026 Max Planck Institute for Security and Privacy (MPI-SP), University of Luebeck Institute for IT-Security (ITS)
from dataclasses import dataclass
from pathlib import Path
import csv
from typing import Dict, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import subprocess

In [ ]:
BENCHMARK_PATH = Path('benchmarks') / Path('20250819-090339') # adjust to your needs

CSV_FILE = BENCHMARK_PATH / 'ever_benchmark.csv'
CONF_FILE = BENCHMARK_PATH / 'ever_benchmark.conf'
LOG_FILE = BENCHMARK_PATH / 'ever_benchmark.log'

LATEX_OUTFILE = BENCHMARK_PATH / 'benchmark_table.tex'
LATEX_OUTFILE_HIGH_D = BENCHMARK_PATH / 'benchmark_table_high_d.tex'

PLOT_TUPLE_OUTFILE = BENCHMARK_PATH / 'plot_tuples.svg'
PLOT_VERIF_TIME_OUTFILE = BENCHMARK_PATH / 'plot_verif_time.svg'

# (gadget, e, k) tuples to plot
SELECTED_SERIES_FOR_PLOT = [
    # ('SWPolySubGadget', 0, 1),
    # ('SWPolySubGadget', 1, 1),
    # ('SWPolySubGadget', 2, 1),
    # ('SWPolySubGadget', 3, 1),
    # ('SecIPRefreshGadget', 0, 1),
    ('SWPolyMulGadget', 0, 1),
    ('SWPolyMulGadget', 1, 1),
    ('SWPolyMulGadget', 2, 1),
    ('SWPolyMulGadget', 3, 1),
]

In [ ]:
@dataclass
class BenchmarkResultRow:
    gadget: str
    d: int
    e: int
    k: int
    field: str
    t: int
    notion: str
    result_correct: bool
    result_secure: bool
    gen_time: float
    check_correctness_time: float
    verif_time: float
    num_tuples: int
    num_threads: int
    comment: str

class BenchmarkResultForGadget:
    name: str
    num_tuples: Dict[Tuple[int, int, int], int] # map (d,e,k) to number of tuples for that gadget
    verif_time: Dict[Tuple[int, int, int], float] = {} # map (d,e,k) to verification time
    comment: Dict[Tuple[int, int, int], str] = {} # map (d,e,k) to comment for that gadget
    
    def __init__(self, name: str):
        self.name = name
        self.num_tuples = {}
        self.verif_time = {}

    def add_result(self, d: int, e: int, k: int, num_tuples: int, verif_time: float, comment: str):
        if (d, e, k) in self.num_tuples:
            raise ValueError(f"Duplicate result for {self.name} with d={d}, e={e}, k={k}")
        self.num_tuples[(d, e, k)] = num_tuples
        self.verif_time[(d, e, k)] = verif_time
        self.comment[(d, e, k)] = comment

In [ ]:
benchmarks = []

with open(CSV_FILE, 'r') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        benchmarks.append(BenchmarkResultRow(
            gadget=row['gadget'],
            d=int(row['d']),
            e=int(row['e']),
            k=int(row['k']),
            field=row.get('field', ''),
            t=int(row['t']),
            notion=row.get('notion', ''),
            result_correct=bool(row['result_correct']),
            result_secure=bool(row['result_secure']),
            gen_time=float(row['gen_time']) if 'gen_time' in row and row['gen_time'] and row['gen_time'] != '' else 0.0,
            check_correctness_time=float(row['check_correctness_time']) if 'check_correctness_time' in row and row['check_correctness_time'] and row['check_correctness_time'] != '' else 0.0,
            verif_time=float(row['verif_time']) if 'verif_time' in row and row['verif_time'] and row['verif_time'] != '' else 0.0,
            num_tuples=int(row['num_tuples']) if 'num_tuples' in row and row['num_tuples'] and row['num_tuples'] != '' else 0,
            num_threads=int(row['num_threads']) if 'num_threads' in row and row['num_threads'] and row['num_threads'] != '' else 0,
            comment=row.get('comment', '')
        ))

gadgets = set(b.gadget for b in benchmarks)

results = {}
for g in gadgets:
    # Extract all benchmarks for the current gadget
    filtered_benchmarks = [b for b in benchmarks if b.gadget == g]

    result = BenchmarkResultForGadget(g)

    for b in filtered_benchmarks:
        # only care about SNI for now
        if b.notion != 'NI':
            continue
        # Print number of tuples for each benchmark
        result.add_result(b.d, b.e, b.k, b.num_tuples, b.verif_time, b.comment)

    results[g] = result

# Convert the results to a DataFrame for further analysis or export
df = pd.DataFrame({
    'gadget': [],
    'd': [],
    'e': [],
    'k': [],
    'num_tuples': [],
    'verif_time': [],
    'comment': [],
})

for gadget, result in results.items():
    for (d, e, k), num_tuples in result.num_tuples.items():
        verif_time = result.verif_time.get((d, e, k), 0)
        comment = result.comment.get((d, e, k), '')
        df = pd.concat([df, pd.DataFrame({
            'gadget': [gadget],
            'd': [d],
            'e': [e],
            'k': [k],
            'num_tuples': [num_tuples],
            'verif_time': [verif_time],
            'comment': [comment]
        })])

# Convert d, e, k, num_tuples to int
df['d'] = df['d'].astype(int)
df['e'] = df['e'].astype(int)
df['k'] = df['k'].astype(int)
df['num_tuples'] = df['num_tuples'].astype(int)

# set the index column to be gadget
df.set_index('gadget', inplace=True)

# merge the index column, i.e., make it a multi-index with d, e, k
df.set_index(['d', 'e', 'k'], append=True, inplace=True)

df_filtered = df.copy()

In [ ]:
gadget_order = ['SWPolyAddGadget', 'SWPolySubGadget', 'SWPolyMulGadget', 'SWComp', 'Refresh', 'OptRefresh', 'RefreshSFRES18', 'LaolaMultGadget', 'IPAddGadget', 'IPMultGadget', 'IPRefreshGadget', 'SecIPRefreshGadget']

In [ ]:
def num2str(n: int) -> str:
    if n == 0:
        return '-'
    elif n < 1000:
        return str(n)
    elif n < 1000000:
        return f"{n/1000:.1f}k"
    elif n < 1000000000:
        return f"{n/1000000:.1f}M"
    else:
        return f"{n/1000000000:.1f}G"

In [ ]:
# Generate a latex table with columns: Component, e=0, e=1, e=2. Each e has three subcolumns: d=1, d=2, d=3. Component is the gadget name and should be centered vertically
# The table headers should be centered and the values should be aligned to the right.
# The table should have a top rule, a mid rule after the headers, and a bottom rule at the end.
latex_table = r"""\documentclass{standalone}
\usepackage{booktabs}
\usepackage{multirow}
\usepackage{amsmath}
\usepackage{amsfonts}
\usepackage{amssymb}
\usepackage[table]{xcolor}
\rowcolors{1}{white}{gray!30}
\begin{document}
\begin{tabular}{lrrrrrrrrr}
\toprule
\multirow{2}{*}{Gadget} & & \(e=0\) & & & \(e=1\) & & & \(e=2\) \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10}
    & \(d=1\) & \(d=2\) & \(d=3\) & \(d=1\) & \(d=2\) & \(d=3\) & \(d=1\) & \(d=2\) & \(d=3\) \\
\midrule
"""
for gadget in gadget_order:
    result = results.get(gadget, None)
    latex_table += f"{gadget} &"
    for e in range(3):  # e = 0, 1, 2
        if gadget in ("IPAddGadget", "SecIPRefreshGadget", "IPMultGadget", "IPRefreshGadget") and e > 0:
            continue
        for d in range(1, 4):  # d = 1, 2, 3
            verif_time = result.verif_time.get((d, e, 1), 0)
            if verif_time >= 899.5 and result.comment.get((d, e, 1), '') == 'Timeout':
                latex_table += r"\( \bot \) &" 
            else:
                latex_table += f" {verif_time:3.0f}s &" if verif_time > 1.0 else " - &" if verif_time == 0.0 else r" \( \varepsilon \) &"
    latex_table = latex_table[:-2] + r"\\" + "\n" + r"\quad \# tuples &"
    for e in range(3):  # e = 0, 1, 2
        if gadget in ("IPAddGadget", "SecIPRefreshGadget", "IPMultGadget", "IPRefreshGadget") and e > 0:
            continue
        for d in range(1, 4):  # d = 1, 2, 3
            num_tuples = num2str(result.num_tuples.get((d, e, 1), 0)) # k=1 for simplicity
            latex_table += f" {num_tuples} &"
    latex_table = latex_table[:-2] + r"\\" + "\n"
latex_table += r"""\bottomrule
\end{tabular}
\end{document}
"""

In [ ]:
# same as above, but for d=4,5,6

# Generate a latex table with columns: Component, e=0, e=1, e=2. Each e has three subcolumns: d=4, d=5, d=6. Component is the gadget name and should be centered vertically
# The table headers should be centered and the values should be aligned to the right.
# The table should have a top rule, a mid rule after the headers, and a bottom rule at the end.
latex_table_high_d = r"""\documentclass{standalone}
\usepackage{booktabs}
\usepackage{multirow}
\usepackage{amsmath}
\usepackage{amsfonts}
\usepackage{amssymb}
\usepackage[table]{xcolor}
\rowcolors{1}{white}{gray!30}
\begin{document}
\begin{tabular}{lrrrrrrrrr}
\toprule
\multirow{2}{*}{Gadget} & & \(e=0\) & & & \(e=1\) & & & \(e=2\) \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10}
    & \(d=4\) & \(d=5\) & \(d=6\) & \(d=4\) & \(d=5\) & \(d=6\) & \(d=4\) & \(d=5\) & \(d=6\) \\
\midrule
"""
for gadget in gadget_order:
    result = results.get(gadget, None)
    if not result:
        continue
    latex_table_high_d += f"{gadget} &"
    for e in range(3):  # e = 0, 1, 2
        if gadget in ("IPAddGadget", "SecIPRefreshGadget", "IPMultGadget", "IPRefreshGadget") and e > 0:
            continue
        for d in range(4, 7):  # d = 4, 5, 6
            verif_time = result.verif_time.get((d, e, 1), 0)
            if verif_time >= 899.5 or result.comment.get((d, e, 1), '') == 'Timeout':
                latex_table_high_d += r" \( \bot \) &" 
            elif result.comment.get((d,e,1),'') == 'Skipped':
                latex_table_high_d += r" - &"
            else:
                latex_table_high_d += f" {verif_time:3.0f}s &" if float(verif_time) > 1.0 else " - &" if float(verif_time) == 0.0 else r" \( \varepsilon \) &"
    latex_table_high_d = latex_table_high_d[:-2] + r"\\" + "\n" + r"\quad \# tuples &"
    for e in range(3):  # e = 0, 1, 2
        if gadget in ("IPAddGadget", "SecIPRefreshGadget", "IPMultGadget", "IPRefreshGadget") and e > 0:
            continue
        for d in range(4, 7):  # d = 4, 5, 6
            num_tuples = num2str(result.num_tuples.get((d, e, 1), 0)) # k=1 for simplicity
            latex_table_high_d += f" {num_tuples} &"
    latex_table_high_d = latex_table_high_d[:-2] + r"\\" + "\n"
latex_table_high_d += r"""\bottomrule
\end{tabular}
\end{document}
"""

In [ ]:
# Write latex to a file and compile it to PDF
with open(LATEX_OUTFILE, 'w') as f:
    f.write(latex_table)
    print("LaTeX table saved to benchmark_results.tex")

# Write latex to a file and compile it to PDF
with open(LATEX_OUTFILE_HIGH_D, 'w') as f:
    f.write(latex_table_high_d)
    print("LaTeX table saved to benchmark_results_2.tex")

In [ ]:
# Compile the LaTeX file to PDF
subprocess.run(['pdflatex', f'-output-directory={BENCHMARK_PATH}', str(LATEX_OUTFILE)], check=True)
# Clean up auxiliary files generated by LaTeX
aux_files = [LATEX_OUTFILE.with_suffix(suffix) for suffix in ['.aux', '.log', '.out']]
for aux_file in aux_files:
    if aux_file.exists():
        aux_file.unlink()
        print(f"Removed auxiliary file: {aux_file}")
print("LaTeX compilation completed. Check the PDF for results.")

# Compile the LaTeX file to PDF
subprocess.run(['pdflatex', f'-output-directory={BENCHMARK_PATH}', str(LATEX_OUTFILE_HIGH_D)], check=True)
# Clean up auxiliary files generated by LaTeX
aux_files = [LATEX_OUTFILE_HIGH_D.with_suffix(suffix) for suffix in ['.aux', '.log', '.out']]
for aux_file in aux_files:
    if aux_file.exists():
        aux_file.unlink()
        print(f"Removed auxiliary file: {aux_file}")
print("LaTeX compilation completed. Check the PDF for results.")

In [ ]:
plt.figure(figsize=(10, 6))
plt.yscale('log')
# Add a legend entry for the gadget
plt.xlabel('d (degree)')
plt.ylabel('# tuples to verify')
plt.title('Number of tuples vs degree (d) for each gadget')
plt.xticks(np.arange(1, 7))

# set index to be gadget, d, e, k
#print(df_filtered)
df_plot_filtered = df_filtered.copy()
df_plot_filtered.reset_index(inplace=True)
df_plot_filtered.set_index(['gadget', 'e', 'k'], inplace=True)
#print(df_plot_filtered)

# plot each desired series based on df_filtered
for gadget, e, k in SELECTED_SERIES_FOR_PLOT:
    # retrieve all the data for the gadget, e, k
    data = df_plot_filtered.loc[gadget].loc[e].loc[k]
    # retrieve all (d, num_tuples) pairs
    data = data[['d', 'num_tuples']].set_index('d')
    # plot the data
    plt.scatter(data.index, data['num_tuples'], label=f'{gadget} (e={e}, k={k})', marker='+')

plt.legend()
plt.grid(True, which='major', linestyle='dashed', axis='y', linewidth=0.5)
plt.tight_layout()
plt.savefig(PLOT_TUPLE_OUTFILE)
plt.show()

In [ ]:
# Same as before, but this time with verification time
plt.figure(figsize=(10, 6))
plt.yscale('log')
plt.xlabel('d (degree)')
plt.ylabel('Verification time (s)')
plt.title('Verification time vs degree (d) for each gadget')
plt.xticks(np.arange(1, 7))
# plot each desired series based on df_filtered
for gadget, e, k in SELECTED_SERIES_FOR_PLOT:
    # retrieve all the data for the gadget, e, k
    data = df_plot_filtered.loc[gadget].loc[e].loc[k]
    # retrieve all (d, verif_time) pairs
    data = data[['d', 'verif_time']].set_index('d')
    # plot the data
    plt.scatter(data.index, data['verif_time'], label=f'{gadget} (e={e}, k={k})', marker='+')
plt.legend()
plt.grid(True, which='major', linestyle='dashed', axis='y', linewidth=0.5)
plt.tight_layout()
plt.savefig(PLOT_VERIF_TIME_OUTFILE)
plt.show()